In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, accuracy_score

import joblib


In [14]:
try:
    df = pd.read_csv("indian_climate.csv")
except:
    print("Dataset file not found. Please add weather.csv to run the project.")

In [15]:
df['date'] = pd.to_datetime(df['date'])

df['day_of_year'] = df['date'].dt.dayofyear
df['year'] = df['date'].dt.year

In [16]:
le = LabelEncoder()
df['city_encoded'] = le.fit_transform(df['city'])


In [17]:
def heatwave_risk(temp):
    if temp >= 42: return 2
    elif temp >= 40: return 1
    else: return 0

df['heatwave_risk'] = df['temperature_2m_max'].apply(heatwave_risk)


In [18]:
X_reg = df[['city_encoded','day_of_year','year']]
y_reg = df['temperature_2m_max']

X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_model = RandomForestRegressor(n_estimators=200, random_state=42)
reg_model.fit(X_train, y_train)

print("Temp RMSE:", np.sqrt(((reg_model.predict(X_test)-y_test)**2).mean()))


Temp RMSE: 1.1870677096425724


In [19]:
# Predict temperature for all rows using Stage 1 model
df['predicted_temp'] = reg_model.predict(X_reg)
print(df[['temperature_2m_max','predicted_temp']])


       temperature_2m_max  predicted_temp
0                    19.9         19.7940
1                    20.0         19.9175
2                    20.1         20.0095
3                    19.8         19.7980
4                    19.4         19.5030
...                   ...             ...
91315                25.6         24.5470
91316                22.4         23.3045
91317                22.8         22.4720
91318                20.8         21.2585
91319                18.3         21.2105

[91320 rows x 2 columns]


In [20]:
# --------- CLASSIFICATION (Stage 2) ----------

# Features (use predicted temperature, NOT real temperature)
X_cls = df[['predicted_temp',
            'wind_speed_10m_max',
            'wind_gusts_10m_max',
            'precipitation_sum',
            'rain_sum',
            'weather_code']]

y_cls = df['heatwave_risk']

# Train–test split
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# --------- BALANCE TRAIN DATA ----------
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# --------- MODEL ----------
cls_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

cls_model.fit(X_train_bal, y_train_bal)

# --------- EVALUATION ----------
y_pred = cls_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.982862461673237
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     17550
           1       0.65      0.82      0.73       494
           2       0.78      0.82      0.80       220

    accuracy                           0.98     18264
   macro avg       0.81      0.88      0.84     18264
weighted avg       0.99      0.98      0.98     18264



In [21]:
joblib.dump(reg_model, "temp_forecast_model.pkl")
joblib.dump(cls_model, "heatwave_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "city_encoder.pkl")


['city_encoder.pkl']

In [22]:
# def predict_heatwave(city, future_date):
#     reg = joblib.load("temp_forecast_model.pkl")
#     cls = joblib.load("heatwave_model.pkl")
#     scaler = joblib.load("scaler.pkl")
#     le = joblib.load("city_encoder.pkl")

#     date = pd.to_datetime(future_date)
#     doy = date.dayofyear
#     year = date.year
#     city_enc = le.transform([city])[0]

#     # Stage 1: predict temperature
#     temp_pred = reg.predict([[city_enc, doy, year]])[0]

#     # dummy average values for now
#     wind = 10; gust = 20; precip = 0; rain = 0; code = 1

#     X = scaler.transform([[temp_pred, wind, gust, precip, rain, code]])
#     risk = cls.predict(X)[0]

#     return temp_pred, ["Low","Medium","High"][risk]


In [23]:
#print(predict_heatwave("Delhi", "2026-05-20"))
